# Kafka to PostgreSQL Ingestion Workshop
This notebook demonstrates how to consume streaming data from a Kafka topic and persist it into a PostgreSQL database.

### Step 1: Library Imports
Import essential libraries for environment management, data processing, and database connectivity.

In [55]:
import os
import sys
import json
import psycopg2
from kafka import KafkaConsumer
from datetime import datetime

### Step 2: Environment Setup
Configure project paths to enable importing custom models from the `src` directory.

In [56]:
# Add the project root (parent of notebooks/) to sys.path
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import custom data models
from src.models import GreenRide, ride_deserializer

### Step 3: Kafka Configuration
Initialize the connection parameters for the Kafka broker.

In [57]:
topic_name = 'green-trips'
bootstrap_servers = ['localhost:9092']

### Step 4: Initialize Kafka Consumer
Set up the consumer with a persistent group ID and offset reset policies.

In [58]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=bootstrap_servers,
    auto_offset_reset='earliest',       # Ingest all historical data in the topic
    value_deserializer=ride_deserializer,
    consumer_timeout_ms=5000           # 👈 STOP AUTOMATICALLY AFTER 5 SECONDS OF NO DATA
)

### Step 5: Connect to PostgreSQL
Establish the session with the PostgreSQL container on the default port 5432.

In [59]:
conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="postgres"
)
cur = conn.cursor()
print("✅ Connected to Postgres successfully!")

✅ Connected to Postgres successfully!


### Step 6: Database Initialization
Ensure the target schema exists before starting the ingestion process.

In [60]:
create_table_query = """
CREATE TABLE IF NOT EXISTS green_trips (
    lpep_pickup_datetime TIMESTAMP,
    lpep_dropoff_datetime TIMESTAMP,
    PULocationID INTEGER,
    DOLocationID INTEGER,
    passenger_count INTEGER,
    trip_distance FLOAT,
    tip_amount FLOAT,
    total_amount FLOAT
);
"""
cur.execute(create_table_query)
conn.commit()
print("✅ Postgres table 'green_trips' is ready.")

✅ Postgres table 'green_trips' is ready.


### Step 7: Data Ingestion Loop
The main pipeline: consume from Kafka → Transform → Write to Postgres.

In [66]:
import time

print(f"📡 Listening for data on topic '{topic_name}'...")

cur.execute("TRUNCATE TABLE green_trips")
conn.commit()
print("🧹 Table cleared for fresh run.")

count = 0
batch_size = 10000  # 🚀 Commit every 10k records for MASSIVE speed
t0 = time.time()

try:
    for message in consumer:
        ride = message.value
        
        # 1. Transform Milliseconds -> Datetime
        try:
            pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
            dropoff_dt = datetime.fromtimestamp(ride.lpep_dropoff_datetime / 1000)
        except (TypeError, ValueError):
            pickup_dt = ride.lpep_pickup_datetime
            dropoff_dt = ride.lpep_dropoff_datetime

        # 2. Persist to Database
        insert_query = """
        INSERT INTO green_trips (
            lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, DOLocationID, 
            passenger_count, trip_distance, tip_amount, total_amount
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """
        
        cur.execute(insert_query, (
            pickup_dt, dropoff_dt, ride.PULocationID, ride.DOLocationID,
            ride.passenger_count, ride.trip_distance, ride.tip_amount, ride.total_amount
        ))
        
        count += 1
        
        # 3. Performance: Batch Commit
        if count % batch_size == 0:
            conn.commit()
            print(f"-> [BATCH COMMIT] Ingested {count} records...")
        elif count % 100 == 0:
            # Still show progress every 100 rows
            print(f"-> Ingested {count} records...", end="\r")
            
except KeyboardInterrupt:
    print("\n🛑 Ingestion manually stopped.")
finally:
    # 4. Final Cleanup: Persist the last batch
    conn.commit()
    t1 = time.time()
    duration = t1 - t0
    print(f"Done. Total processed: {count} records.")
    print(f"Total ingestion time: {duration:.2f} seconds")

📡 Listening for data on topic 'green-trips'...
🧹 Table cleared for fresh run.
-> [BATCH COMMIT] Ingested 10000 records...
-> [BATCH COMMIT] Ingested 20000 records...
-> [BATCH COMMIT] Ingested 30000 records...
-> [BATCH COMMIT] Ingested 40000 records...
Done. Total processed: 49416 records.
Total ingestion time: 551.74 seconds


### Step 8: Homework Verification
Count how many trips have a trip_distance greater than 5.0 kilometers.

In [67]:
cur.execute("""
    SELECT count(*) 
    FROM green_trips 
    WHERE trip_distance > 5.0
""")
count_gt_5 = cur.fetchone()[0]
print(f"How many trips have trip_distance > 5? -> {count_gt_5}")

How many trips have trip_distance > 5? -> 8506


### Step 9: Resource Cleanup
Close the Kafka consumer and database connections.

In [ ]:
consumer.close()
cur.close()
conn.close()
print("Resources closed.")